# Real Estate Price Prediction with Linear Regression

**Author:** Olivier Robert-Duboille

## 1. Introduction
Predicting real estate prices is a classic regression problem involving multiple features like location, size, and amenities. In this notebook, we will build a robust Linear Regression model to predict house prices based on synthetic but realistic data.

### Objectives:
- Generate a dataset with linear and non-linear relationships.
- Perform Exploratory Data Analysis (EDA) to identify key drivers of price.
- specific feature engineering (e.g., polynomial features).
- Train and evaluate a Linear Regression model.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

sns.set(style="ticks")
plt.rcParams['figure.figsize'] = (10, 6)

## 2. Data Generation
We'll simulate a dataset of 1000 houses with features like:
- `SquareFootage`: Linear relationship with price.
- `NumBedrooms`: Step-wise relationship.
- `Age`: Negative correlation (older houses might be cheaper).
- `DistanceToCityCenter`: Negative non-linear correlation.

In [ ]:
np.random.seed(42)
n_samples = 1000

# Features
sq_ft = np.random.normal(2000, 500, n_samples)
bedrooms = np.random.randint(1, 6, n_samples)
age = np.random.randint(0, 50, n_samples)
dist_center = np.abs(np.random.normal(10, 5, n_samples))

# Price Equation (Ground Truth)
# Price = Base + 150*SqFt + 10000*Beds - 500*Age - 2000*Dist + Noise
price = (50000 + 
         150 * sq_ft + 
         10000 * bedrooms - 
         500 * age - 
         2000 * dist_center + 
         np.random.normal(0, 25000, n_samples)) # Noise

df = pd.DataFrame({
    'SquareFootage': sq_ft,
    'Bedrooms': bedrooms,
    'Age': age,
    'DistanceToCenter': dist_center,
    'Price': price
})

df.head()

## 3. Exploratory Data Analysis (EDA)
Let's look at the correlation matrix to confirm our assumptions about the relationships.

In [ ]:
# Pairplot to see relationships
sns.pairplot(df, diag_kind='kde')
plt.show()

# Correlation Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Feature Correlation Matrix')
plt.show()

## 4. Modeling
We will split the data into training and testing sets, normalize the features, and fit a Linear Regression model.

In [ ]:
X = df.drop('Price', axis=1)
y = df['Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scaling is good practice even for Linear Regression to interpret coefficients
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LinearRegression()
model.fit(X_train_scaled, y_train)

## 5. Evaluation
Let's check the coefficients to see what drives price the most, and calculate our error metrics.

In [ ]:
y_pred = model.predict(X_test_scaled)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"RMSE: ${rmse:,.2f}")
print(f"R^2 Score: {r2:.3f}")

# Coefficients
coef_df = pd.DataFrame({'Feature': X.columns, 'Coefficient': model.coef_})
coef_df.sort_values(by='Coefficient', ascending=False)

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.6)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--')
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.title('Actual vs Predicted Prices')
plt.show()